# Week 4 — Baseline Linear Regression Model

In this notebook, I trained a baseline Linear Regression model to predict California residential single-family property close prices. The goal of this baseline model is to create an initial performance benchmark before testing more advanced models in later weeks.


In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

I imported pandas and numpy for data handling, LinearRegression for the baseline model, and evaluation metrics to measure model performance on the test set.

In [2]:
train_df = pd.read_csv("train_week3_X12_months.csv")
test_df = pd.read_csv("test_week3_2026-05.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (129867, 57)
Test shape: (12024, 57)


In [3]:
numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "YearBuilt",
    "DaysOnMarket",
    "GarageSpaces",
    "ParkingTotal",
    "OriginalListPrice",
    "ListPrice",
    "Latitude",
    "Longitude",
    "Stories",
    "AssociationFee",
    "MainLevelBedrooms"
]

categorical_features = [
    "CountyOrParish",
    "City",
    "PostalCode",
    "HighSchoolDistrict",
    "Flooring",
    "AttachedGarageYN",
    "Levels",
    "PoolPrivateYN",
    "NewConstructionYN",
    "ViewYN",
    "FireplaceYN"
]

In [4]:
feature_numeric = numeric_features.copy()

for col in ["home_age", "log_LivingArea", "log_LotSizeSquareFeet"]:
    if col in train_df.columns:
        feature_numeric.append(col)

missing_flag_cols = [col for col in train_df.columns if col.endswith("_missing_flag")]
feature_numeric += missing_flag_cols

feature_numeric = list(dict.fromkeys([col for col in feature_numeric if col in train_df.columns]))
feature_categorical = [col for col in categorical_features if col in train_df.columns]

print("Number of numeric predictors:", len(feature_numeric))
print("Number of categorical predictors:", len(feature_categorical))

Number of numeric predictors: 38
Number of categorical predictors: 11


In [5]:
X_train_raw = train_df[feature_numeric + feature_categorical].copy()
X_test_raw = test_df[feature_numeric + feature_categorical].copy()

y_train = train_df["ClosePrice"].copy()
y_test = test_df["ClosePrice"].copy()

print("X_train_raw shape:", X_train_raw.shape)
print("X_test_raw shape:", X_test_raw.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train_raw shape: (129867, 49)
X_test_raw shape: (12024, 49)
y_train shape: (129867,)
y_test shape: (12024,)


In [15]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Make categorical columns strings
for col in feature_categorical:
    X_train_raw[col] = X_train_raw[col].astype(str)
    X_test_raw[col] = X_test_raw[col].astype(str)

# Use sparse output to save memory
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=True
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse=True
    )

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), feature_numeric),
        ("cat", encoder, feature_categorical)
    ],
    remainder="drop"
)

X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)
print("Data type:", type(X_train_processed))

Processed X_train shape: (129867, 3890)
Processed X_test shape: (12024, 3890)
Data type: <class 'scipy.sparse._csr.csr_matrix'>


/Users/nguyenanh/miniforge3/envs/dsc80/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [1, 2, 3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


#### **Train the Linear Regression model**

In [7]:
baseline_model = LinearRegression()

baseline_model.fit(X_train_processed, y_train)

print("Baseline Linear Regression model trained.")

Baseline Linear Regression model trained.


#### **Predict on the test set**

In [8]:
y_pred = baseline_model.predict(X_test_processed)

print("Number of predictions:", len(y_pred))
print("First 5 predictions:", y_pred[:5])

Number of predictions: 12024
First 5 predictions: [ 930816.  540672. 9371648.  402432. 2025472.]


#### **Evaluate the model**

In [9]:
r2 = r2_score(y_test, y_pred)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Baseline Linear Regression Results")
print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

Baseline Linear Regression Results
R²: -1.4453033458181543e+20
MAE: 893748905512159.1
RMSE: 2.0173768018236956e+16


In [10]:
baseline_results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "Training Window": ["12 months"],
    "Test Month": ["2026-05"],
    "R2": [r2],
    "MAE": [mae],
    "RMSE": [rmse]
})

baseline_results

,Model,Training Window,Test Month,R2,MAE,RMSE
0,Linear Regression,12 months,2026-05,-1.445303e+20,8.937489e+14,2.017377e+16


In [11]:
print("Actual ClosePrice range:")
print("Min:", y_test.min())
print("Max:", y_test.max())
print("Mean:", y_test.mean())

print("\nPredicted ClosePrice range:")
print("Min:", y_pred.min())
print("Max:", y_pred.max())
print("Mean:", y_pred.mean())

pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10]
})

Actual ClosePrice range:
Min: 11900.0
Max: 97972500.0
Mean: 1309788.5996091152

Predicted ClosePrice range:
Min: -1.1417018965158963e+18
Max: 2.415047209103831e+17
Mean: -348864749639563.9


,Actual,Predicted
0,885000.0,930816.0
1,525000.0,540672.0
2,9700000.0,9371648.0
3,315000.0,402432.0
4,2100000.0,2025472.0
5,645500.0,717824.0
6,400000.0,557056.0
7,1045000.0,973824.0
8,1385000.0,1459200.0
9,761125.0,730112.0


In [12]:
error_check = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Error": y_pred - y_test.values,
    "Absolute_Error": np.abs(y_pred - y_test.values)
})

error_check.sort_values("Absolute_Error", ascending=False).head(20)

,Actual,Predicted,Error,Absolute_Error
6256,450000.0,-1.141702e+18,-1.141702e+18,1.141702e+18
7574,400000.0,-1.141702e+18,-1.141702e+18,1.141702e+18
7656,400000.0,-8.392172e+17,-8.392172e+17,8.392172e+17
3057,650000.0,-6.649767e+17,-6.649767e+17,6.649767e+17
10281,584000.0,-4.103317e+17,-4.103317e+17,4.103317e+17
9480,1400000.0,-4.038549e+17,-4.038549e+17,4.038549e+17
1098,365000.0,-3.631911e+17,-3.631911e+17,3.631911e+17
10071,541000.0,-2.558512e+17,-2.558512e+17,2.558512e+17
2836,390000.0,2.415047e+17,2.415047e+17,2.415047e+17
11341,519000.0,2.415047e+17,2.415047e+17,2.415047e+17


In [16]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=1.0, solver="lsqr")

ridge_model.fit(X_train_processed, y_train)

ridge_pred = ridge_model.predict(X_test_processed)

print("Number of predictions:", len(ridge_pred))
print("First 5 predictions:", ridge_pred[:5])

Number of predictions: 12024
First 5 predictions: [ 946866.71063474  562109.32606205 9511082.41580604  312968.95881762
 2059620.17677441]


In [18]:
print("Actual ClosePrice range:")
print("Min actual:", y_test.min())
print("Max actual:", y_test.max())
print("Mean actual:", y_test.mean())

print("\nRidge Predicted ClosePrice range:")
print("Min predicted:", ridge_pred.min())
print("Max predicted:", ridge_pred.max())
print("Mean predicted:", ridge_pred.mean())

Actual ClosePrice range:
Min actual: 11900.0
Max actual: 97972500.0
Mean actual: 1309788.5996091152

Ridge Predicted ClosePrice range:
Min predicted: -5599401.202121839
Max predicted: 31670965.25663273
Mean predicted: 1407337.1419757565


In [19]:
ridge_r2 = r2_score(y_test, ridge_pred)
ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

print("Ridge Regression Results")
print("R²:", ridge_r2)
print("MAE:", ridge_mae)
print("RMSE:", ridge_rmse)

Ridge Regression Results
R²: 0.43672979984954896
MAE: 299605.71023416286
RMSE: 1259406.429706948


In [20]:
baseline_results = pd.DataFrame({
    "Model": ["Plain Linear Regression", "Ridge Regression"],
    "Training Window": ["12 months", "12 months"],
    "Test Month": ["2026-05", "2026-05"],
    "R2": [r2, ridge_r2],
    "MAE": [mae, ridge_mae],
    "RMSE": [rmse, ridge_rmse]
})

baseline_results

,Model,Training Window,Test Month,R2,MAE,RMSE
0,Plain Linear Regression,12 months,2026-05,-1.445303e+20,8.937489e+14,2.017377e+16
1,Ridge Regression,12 months,2026-05,4.367298e-01,2.996057e+05,1.259406e+06


Although Ridge Regression stabilized the baseline model, it can still produce negative predictions because linear models do not naturally restrict predicted prices to be positive.

The initial plain Linear Regression model produced extremely unstable results, including very large positive and negative predictions. This caused the R², MAE, and RMSE values to become unrealistic. After checking the largest prediction errors, I found that some predicted prices were on the scale of 10^17 to 10^18, even though the actual prices were normal.

To create a more stable linear baseline, I tested Ridge Regression. Ridge Regression is a regularized version of Linear Regression that helps reduce extreme coefficient values. The Ridge model produced a more reasonable baseline result, with an R² score of 0.437 on the May 2026 test set. The model had an MAE of about $299,606 and an RMSE of about $1,259,406. This baseline will be used as the comparison point for more advanced models in Week 5.

## Conclusion

For Week 4, I trained a baseline linear model to predict ClosePrice for residential single-family homes in California. The plain Linear Regression model was unstable due to the large number of one-hot encoded categorical features, so I used Ridge Regression as a more stable linear baseline. The Ridge model achieved an R² score of 0.437 on the May 2026 test set. This result provides the first benchmark for future models, including Decision Tree and Random Forest regressors in Week 5.